# Forecasting price as a range

**Time-series features · version 4 · 11 November 2026.** Builds on versions 2 and 3 (`time_series_features_revisited.ipynb`). Version 1 is `time_series.ipynb` (git tag `v1`), unchanged. History: `VERSIONS.md`.

The time-series revision (`time_series_features_revisited.ipynb`) ended on three findings: risk is predictable, which stock will outperform is not, and the market's own return is weakly higher after fear spikes. This notebook turns those into something usable:

| Section | Question |
|---|---|
| 1 | Does the fear-and-return finding hold on periods it was not found on (1990–2009 and 2022–2026)? |
| 2 | Can a model forecast next month's volatility better than the trailing volatility the original features used? |
| 3 | Is the resulting price range honest: do 90% of outcomes land inside the 90% band? |
| 4 | Does tilting the centre of the range by market state improve it? |
| 5 | Does sizing a position by forecast risk beat simply holding? |

A range is the form of price forecast the evidence supports: not "up or down", but "between here and here, with this confidence".

**Rules fixed in advance.** The confirmation rule for section 1, the four range methods, the three strategies and how each is scored were written to `data_cache/vix_confirmation_prereg.json` before the new data was downloaded or any of this was run.

**Data.** Section 1: the S&P 500 index and VIX from 1990 to November 2026 (index level only). Sections 2 to 5: the frozen S&P 500 stock snapshot ending December 2021, each company counted only from the date it joined the index; stock-level data after 2021 is held back for another project's final test.

**Known weakness.** The features in the volatility model were chosen after seeing 2010–2021, the same years it is scored on. The model is refitted on past data only, so its weights never see the future, but the choice of inputs did.

In [1]:
import json
import os
import sys

import numpy as np
import pandas as pd
import plotly.graph_objects as go

for path in (os.getcwd(), os.path.abspath(os.path.join(os.getcwd(), '..'))):
    if os.path.exists(os.path.join(path, 'func_lib.py')) and path not in sys.path:
        sys.path.append(path)
import cross_section as cs
import func_lib
import range_forecast as rf_
import ts_features as tf
import vol_eval as ve

H = 21                       # trading days: the forecast horizon
FIRST_OOS = '2012-01-01'     # first out-of-sample day for the stock-level models (two years to fit on)
END = '2021-12-31'
TICKER = 'AAPL'
COST_BPS = 5.0

## 1. Confirming the fear-and-return finding on new periods

Found on 2010–2021: when VIX is high against its own history so far, the next 21 days' market return is higher. The rule written down beforehand: it is **confirmed** only if the correlation is positive in *both* new periods and its t-statistic exceeds 2 on the two pooled.

In [2]:
idx_data = pd.read_csv(os.path.join('data_cache', 'spx_vix_1990_2026.csv'), index_col=0, parse_dates=True).dropna(subset=['^GSPC', '^VIX'])
spx_fwd = np.log(idx_data['^GSPC'].shift(-H) / idx_data['^GSPC'])
vix_state = tf.market_state(idx_data['^VIX'], idx_data['^VIX3M'])
PERIODS = {'1990-2009 (new)': ('1991-01-01', '2009-12-31'), '2010-2021 (where it was found)': ('2010-01-01', '2021-12-31'),
           '2022-2026 (new)': ('2022-01-01', '2026-12-31')}


def corr_test(x, y):
    return tf.predictive_table(x.to_frame('x'), y, H).loc['x']


rows = {}
for name, (a, b) in PERIODS.items():
    r = corr_test(vix_state['vix_pct'].loc[a:b], spx_fwd.loc[a:b])
    rows[name] = {'rank correlation': r['corr'], 't': r['t'], 'days': int(r['days'])}
new = pd.concat([vix_state['vix_pct'].loc[a:b] for n, (a, b) in PERIODS.items() if 'new' in n])
r = corr_test(new, spx_fwd.reindex(new.index))
rows['both new periods pooled'] = {'rank correlation': r['corr'], 't': r['t'], 'days': int(r['days'])}
confirm = pd.DataFrame(rows).T
confirmed = bool(confirm.loc['1990-2009 (new)', 'rank correlation'] > 0 and confirm.loc['2022-2026 (new)', 'rank correlation'] > 0
                 and confirm.loc['both new periods pooled', 't'] > 2)
print('VIX against its own history vs the next-21-day S&P 500 return:')
print(confirm.round(3).to_string())
print(f"\nRule: positive in both new periods and t > 2 pooled  ->  {'CONFIRMED' if confirmed else 'NOT CONFIRMED'}")

slope = corr_test(vix_state['vix_slope'].loc['2022-01-01':], spx_fwd.loc['2022-01-01':])
print(f"Secondary, term-structure slope on 2022-2026: rank correlation {slope['corr']:+.3f} (t {slope['t']:.1f}); found sign was negative.")

# What it is worth: average next-21-day return by third of the VIX state, over the whole history
third = pd.cut(vix_state['vix_pct'], [0, 1 / 3, 2 / 3, 1.0], labels=['calm', 'normal', 'fearful'])
by_third = pd.DataFrame({'mean next-21d return %': 100 * spx_fwd.groupby(third, observed=True).mean(),
                         'share of periods up %': 100 * (spx_fwd > 0).groupby(third, observed=True).mean(),
                         'worst 5% of periods, avg %': 100 * spx_fwd.groupby(third, observed=True).apply(lambda s: s[s <= s.quantile(0.05)].mean()),
                         'days': spx_fwd.groupby(third, observed=True).count()})
print('\n1991-2026, by where VIX stood:')
print(by_third.round(2).to_string())

VIX against its own history vs the next-21-day S&P 500 return:
                                rank correlation      t    days
1990-2009 (new)                            0.104  2.099  4790.0
2010-2021 (where it was found)             0.261  4.639  3021.0
2022-2026 (new)                            0.077  0.712  1198.0
both new periods pooled                    0.096  2.160  5988.0

Rule: positive in both new periods and t > 2 pooled  ->  CONFIRMED
Secondary, term-structure slope on 2022-2026: rank correlation -0.200 (t -2.4); found sign was negative.

1991-2026, by where VIX stood:
         mean next-21d return %  share of periods up %  worst 5% of periods, avg %  days
vix_pct                                                                                 
calm                       0.57                  64.07                       -6.02  3354
normal                     0.54                  64.39                      -10.73  2637
fearful                    1.10                  63.92  

## 2. A volatility model against the trailing number

The original features carried one volatility: the standard deviation of the last 20 days' returns. The model below predicts the next month's volatility for every stock from five inputs that passed the earlier test, with one set of weights for all stocks, refitted each January on past data only.

In [3]:
panel = cs.load_sp500_panel(point_in_time=True)
read = lambda f: pd.read_csv(os.path.join('data_cache', f'sp500_ohlcv_{f}_2006_2021.csv.gz'), index_col=0, parse_dates=True).reindex(
    index=panel.close.index, columns=panel.close.columns)
o, hi, lo, c = read('open'), read('high'), read('low'), read('close')
vix = pd.read_csv(os.path.join('data_cache', 'vix_term_2006_2021.csv'), index_col=0, parse_dates=True).reindex(panel.close.index).ffill()
state = tf.market_state(vix['^VIX'], vix['^VIX3M'])

log_ret = np.log(panel.close).diff()
trailing_vol = log_ret.rolling(20).std() * np.sqrt(252)                       # what the original features used
risk = tf.risk_features(o, hi, lo, c)
wide_state = lambda s: pd.DataFrame(np.repeat(s.to_numpy()[:, None], panel.close.shape[1], axis=1), index=panel.close.index, columns=panel.close.columns)
features = {'log_range_vol': np.log(risk['gk_vol'].where(risk['gk_vol'] > 0)),
            'log_downside_vol': np.log(risk['downside_vol'].where(risk['downside_vol'] > 0)),
            'log_trailing_vol': np.log(trailing_vol.where(trailing_vol > 0)),
            'vix_pct': wide_state(state['vix_pct']), 'vix_slope': wide_state(state['vix_slope'])}
fwd_vol = np.sqrt((log_ret ** 2).rolling(H).mean().shift(-H) * 252)
model_vol, weights = rf_.walk_forward_vol_model(features, fwd_vol, panel.member, FIRST_OOS, H)
print('Model weights at each January refit (outcome: log of next-month volatility):')
print(weights.round(3).to_string())

oos = slice(FIRST_OOS, END)
member = panel.member.loc[oos]
vol_rows = {}
for name, f in (('Trailing 20-day volatility', trailing_vol), ('Range-based volatility alone', risk['gk_vol']), ('Model', model_vol)):
    f, y = f.loc[oos].where(member), fwd_vol.loc[oos].where(member)
    ok = f.notna() & y.notna() & (f > 0) & (y > 0)
    ratio = (y.where(ok) ** 2 / f.where(ok) ** 2)
    ic = cs.ic_summary(cs.daily_rank_ic(f, y, panel.sectors, member), H)
    vol_rows[name] = {'QLIKE': float((ratio - np.log(ratio) - 1).stack().mean()), 'mean error (vol pts)': float(100 * (f.where(ok) - y.where(ok)).stack().mean()),
                      'rank IC within sector': ic['IC'], 'stock-days': int(ok.to_numpy().sum())}
vol_table = pd.DataFrame(vol_rows).T
print(f'\nOut of sample, {FIRST_OOS[:4]} to {END[:4]}:')
print(vol_table.round(3).to_string())
# is the model's daily average loss reliably lower than trailing volatility's?
def daily_loss(f):
    f, y = f.loc[oos].where(member), fwd_vol.loc[oos].where(member)
    ratio = y ** 2 / f.where(f > 0) ** 2
    return (ratio - np.log(ratio) - 1).mean(axis=1)
dm = ve.diebold_mariano(daily_loss(model_vol), daily_loss(trailing_vol), H - 1)
print(f"Diebold-Mariano, model vs trailing volatility: {dm['dm']:.1f} (p {dm['p']:.4f}); negative favours the model.")

Model weights at each January refit (outcome: log of next-month volatility):
            const  log_range_vol  log_downside_vol  log_trailing_vol  vix_pct  vix_slope
2012-01-03 -0.160          0.460             0.116             0.198   -0.079     -0.714
2013-01-02 -0.205          0.461             0.106             0.197   -0.045     -0.786
2014-01-02 -0.313          0.480             0.082             0.188    0.050     -0.622
2015-01-02 -0.418          0.478             0.067             0.174    0.129     -0.538
2016-01-04 -0.430          0.457             0.077             0.174    0.121     -0.435
2017-01-03 -0.454          0.451             0.096             0.137    0.130     -0.475
2018-01-02 -0.471          0.462             0.099             0.115    0.141     -0.469
2019-01-02 -0.498          0.464             0.094             0.103    0.139     -0.408
2020-01-02 -0.513          0.467             0.098             0.091    0.136     -0.370
2021-01-04 -0.493          0.431 


Out of sample, 2012 to 2021:
                              QLIKE  mean error (vol pts)  rank IC within sector  stock-days
Trailing 20-day volatility    0.578                -0.051                  0.462    876973.0
Range-based volatility alone  0.450                 0.982                  0.499    876973.0
Model                         0.427                -1.605                  0.508    876973.0
Diebold-Mariano, model vs trailing volatility: -8.1 (p 0.0000); negative favours the model.


## 3. From volatility to a price range, and is the range honest?

A range for the next 21 days' return is `centre ± q × forecast volatility × √(21/252)`. Four ways to build it, in increasing sophistication:

- **M0**: trailing volatility, and `q` from the normal distribution. This is what the original options notebook did.
- **M1**: trailing volatility, with `q` taken from how past outcomes were actually distributed (stock returns have fatter tails than the normal).
- **M2**: the model's volatility, with `q` from past outcomes.
- **M3**: M2 with the centre tilted by the VIX state (section 4).

M0 to M2 centre the range on zero. A range is **honest** if the stated share of outcomes falls inside it. Among honest ranges, narrower is better; the **interval score** combines both (width, plus a penalty for every miss), lower is better.

In [4]:
outcome = np.log(panel.close.shift(-H) / panel.close).where(panel.member)
scale = np.sqrt(H / 252)
z_trailing, z_model = outcome / (trailing_vol * scale), outcome / (model_vol * scale)
q_trailing, q_model = rf_.past_quantiles(z_trailing.loc['2010-01-01':], FIRST_OOS, H), rf_.past_quantiles(z_model, FIRST_OOS, H)
market_fwd = outcome.mean(axis=1)                                             # the average member's next-21-day log return
flat_centre, tilted_centre = rf_.walk_forward_centre(state['vix_pct'].loc['2010-01-01':], market_fwd.loc['2010-01-01':], FIRST_OOS, H)

LEVELS = {0.5: (0.25, 0.75), 0.9: (0.05, 0.95)}
methods = {}
for level, (pl, ph) in LEVELS.items():
    methods[('M0 trailing vol, normal', level)] = rf_.bands(trailing_vol, 0.0, rf_.NORMAL_Z[pl], rf_.NORMAL_Z[ph], H)
    methods[('M1 trailing vol, past quantiles', level)] = rf_.bands(trailing_vol, 0.0, q_trailing[pl], q_trailing[ph], H)
    methods[('M2 model vol, past quantiles', level)] = rf_.bands(model_vol, 0.0, q_model[pl], q_model[ph], H)
    methods[('M3 model vol + VIX-tilted centre', level)] = rf_.bands(model_vol, tilted_centre - flat_centre, q_model[pl], q_model[ph], H)

rows = {}
for (name, level), (lo_b, hi_b) in methods.items():
    s = rf_.score_bands(lo_b.loc[oos].where(member), hi_b.loc[oos].where(member), outcome.loc[oos], level)
    rows.setdefault(name, {}).update({f'{int(100 * level)}% band: inside %': 100 * s['coverage'], f'{int(100 * level)}% band: width %': 100 * s['width'],
                                      f'{int(100 * level)}% band: interval score': 100 * s['interval_score']})
range_table = pd.DataFrame(rows).T
print(f"Out of sample {FIRST_OOS[:4]}-{END[:4]}, {rf_.score_bands(*[b.loc[oos].where(member) for b in methods[('M2 model vol, past quantiles', 0.9)]], outcome.loc[oos], 0.9)['n']:,} stock-days:")
print(range_table.round(2).to_string())

# honesty by market state: the 90% band's coverage when VIX was calm / normal / fearful
st = pd.cut(state['vix_pct'].loc[oos], [0, 1 / 3, 2 / 3, 1.0], labels=['calm', 'normal', 'fearful'])
by_state = {}
for name in ('M0 trailing vol, normal', 'M2 model vol, past quantiles'):
    lo_b, hi_b = methods[(name, 0.9)]
    inside = ((outcome.loc[oos] >= lo_b.loc[oos]) & (outcome.loc[oos] <= hi_b.loc[oos])).where(lo_b.loc[oos].notna() & outcome.loc[oos].notna() & member)
    by_state[name] = {k: float(100 * inside[st == k].stack().mean()) for k in ('calm', 'normal', 'fearful')}
state_table = pd.DataFrame(by_state).T
print('\nShare of outcomes inside the 90% band, by where VIX stood when the forecast was made:')
print(state_table.round(1).to_string())

Out of sample 2012-2021, 805,077 stock-days:
                                  50% band: inside %  50% band: width %  50% band: interval score  90% band: inside %  90% band: width %  90% band: interval score
M0 trailing vol, normal                        49.61               9.67                     18.75               86.55              23.58                     35.95
M1 trailing vol, past quantiles                49.53               9.16                     18.34               89.13              25.73                     36.22
M2 model vol, past quantiles                   48.56               8.48                     18.20               88.23              22.47                     34.75
M3 model vol + VIX-tilted centre               48.30               8.48                     18.25               87.95              22.47                     34.91



Share of outcomes inside the 90% band, by where VIX stood when the forecast was made:
                              calm  normal  fearful
M0 trailing vol, normal       85.6    86.1     89.4
M2 model vol, past quantiles  86.7    88.6     91.0


In [5]:
# The range for one stock through time: AAPL's 90% band (M2) against what happened
lo_b, hi_b = methods[('M2 model vol, past quantiles', 0.9)]
a = pd.DataFrame({'lower': lo_b[TICKER], 'upper': hi_b[TICKER], 'outcome': outcome[TICKER]}).loc[oos].dropna()
fig = go.Figure()
fig.add_scatter(x=a.index, y=a['upper'], line=dict(width=0), showlegend=False, hoverinfo='skip')
fig.add_scatter(x=a.index, y=a['lower'], fill='tonexty', line=dict(width=0), name='90% band, forecast 21 days earlier', fillcolor='rgba(122,156,200,0.25)')
fig.add_scatter(x=a.index, y=a['outcome'], name='Return over the following 21 days', line=dict(color='#e6e6e6', width=1))
fig.update_layout(title=f'{TICKER}: forecast 90% range for the next 21 days, and what followed', height=440, yaxis=dict(title='21-day log return', tickformat='.0%'))
fig.show()
aapl_inside = float(((a['outcome'] >= a['lower']) & (a['outcome'] <= a['upper'])).mean())
print(f"{TICKER}: {100 * aapl_inside:.1f}% of outcomes inside the 90% band over {len(a):,} days; average band width {100 * (a['upper'] - a['lower']).mean():.1f}% of price.")

AAPL: 81.1% of outcomes inside the 90% band over 2,246 days; average band width 22.7% of price.


## 4. Does tilting the centre help?

M3 moves the centre of every stock's range by the market's expected return given the VIX state, relative to its plain average. The fair test is out of sample: does the tilted forecast of the *market's* next-21-day return beat the plain past average? **Out-of-sample R²** is 1 minus the ratio of their squared errors; positive means the tilt helped.

In [6]:
d = pd.concat([market_fwd, flat_centre, tilted_centre], axis=1, keys=['y', 'flat', 'tilted']).loc[oos].dropna()
oos_r2 = 1 - ((d['y'] - d['tilted']) ** 2).sum() / ((d['y'] - d['flat']) ** 2).sum()
dm_c = ve.diebold_mariano((d['y'] - d['tilted']) ** 2, (d['y'] - d['flat']) ** 2, H - 1)
print(f"Market's next-21-day return, {FIRST_OOS[:4]}-{END[:4]}: out-of-sample R-squared of the VIX-tilted centre {100 * oos_r2:+.2f}% "
      f"(Diebold-Mariano {dm_c['dm']:.1f}, p {dm_c['p']:.2f}; negative favours the tilt)")
print(f"Tilt size: the centre moves between {100 * (d['tilted'] - d['flat']).quantile(0.05):+.2f}% and {100 * (d['tilted'] - d['flat']).quantile(0.95):+.2f}% (5th to 95th percentile); "
      f"a typical 90% band is {range_table.loc['M2 model vol, past quantiles', '90% band: width %']:.0f}% wide.")
gain = range_table.loc['M3 model vol + VIX-tilted centre', '90% band: interval score'] - range_table.loc['M2 model vol, past quantiles', '90% band: interval score']
print(f"Effect on the range itself: M3's 90% interval score is {gain:+.3f} against M2 (negative = better).")

Market's next-21-day return, 2012-2021: out-of-sample R-squared of the VIX-tilted centre +0.87% (Diebold-Mariano -0.3, p 0.73; negative favours the tilt)
Tilt size: the centre moves between -1.30% and +1.41% (5th to 95th percentile); a typical 90% band is 22% wide.
Effect on the range itself: M3's 90% interval score is +0.159 against M2 (negative = better).


## 5. Sizing by forecast risk

Three rules for each stock, rebalanced weekly, 5 basis points on every change of exposure, no leverage:

- **Hold**: always fully invested.
- **Trailing**: exposure = min(1, target ÷ trailing volatility).
- **Model**: exposure = min(1, target ÷ model volatility).

The target for each stock is the average of its own volatility forecasts so far (past only). Because exposure is capped at 1, the rules only ever *reduce* a position, so they are compared on risk-adjusted terms.

In [7]:
ret = panel.close.pct_change(fill_method=None).where(panel.member)
target_trailing = trailing_vol.where(panel.member).expanding(252).mean()
target_model = model_vol.expanding(126).mean()
r_trail, w_trail = rf_.sized_returns(ret, trailing_vol, target_trailing, 5, COST_BPS)
r_model, w_model = rf_.sized_returns(ret, model_vol, target_model, 5, COST_BPS)
valid = w_model.loc[oos].notna() & w_trail.loc[oos].notna() & ret.loc[oos].notna()
strategies = {'Hold': ret.loc[oos].where(valid), 'Sized by trailing volatility': r_trail.loc[oos].where(valid), 'Sized by model volatility': r_model.loc[oos].where(valid)}
exposure = {'Hold': 1.0, 'Sized by trailing volatility': float(w_trail.loc[oos].where(valid).stack().mean()), 'Sized by model volatility': float(w_model.loc[oos].where(valid).stack().mean())}

enough = valid.sum() >= 3 * 252                                   # stocks with at least three years in the test
per_stock = {k: rf_.perf_table(v.loc[:, enough]) for k, v in strategies.items()}
strat_table = pd.DataFrame({k: {'Sharpe, average stock': t['sharpe'].mean(), 'worst drawdown, average stock %': 100 * t['max_drawdown'].mean(),
                                'annual return %': 100 * t['ann_return'].mean(), 'annual volatility %': 100 * t['ann_vol'].mean(),
                                'average exposure': exposure[k]} for k, t in per_stock.items()}).T
better = {k: float(100 * (per_stock[k]['sharpe'] > per_stock['Hold']['sharpe']).mean()) for k in strategies if k != 'Hold'}
print(f'{int(enough.sum())} stocks with at least three years, {FIRST_OOS[:4]}-{END[:4]}:')
print(strat_table.round(2).to_string())
print('Share of stocks whose Sharpe ratio improved on holding:', {k: f'{v:.0f}%' for k, v in better.items()})

book = pd.DataFrame({k: v.mean(axis=1) for k, v in strategies.items()})        # equal-weight portfolio of every stock under each rule
book_table = rf_.perf_table(book)
print('\nEqual-weight portfolio of all stocks under each rule:')
print((book_table * [100, 100, 1, 100]).round(2).rename(columns={'ann_return': 'annual return %', 'ann_vol': 'annual volatility %', 'max_drawdown': 'worst drawdown %'}).to_string())
print(f'\n{TICKER}:')
print(pd.DataFrame({k: t.loc[TICKER] for k, t in per_stock.items()}).T.assign(ann_return=lambda x: 100 * x.ann_return, ann_vol=lambda x: 100 * x.ann_vol,
      max_drawdown=lambda x: 100 * x.max_drawdown).round(2).to_string())

# The tension with section 1: exposure is cut exactly when fear, and the reward for bearing it, is highest
w_by_state = w_model.loc[oos].where(valid).mean(axis=1).groupby(st, observed=True).mean()
mkt_by_state = (100 * market_fwd.loc[oos]).groupby(st, observed=True).mean()
tension = pd.DataFrame({'average exposure (model rule)': w_by_state, 'market return over the next 21 days %': mkt_by_state})
print('\nBy where VIX stood:')
print(tension.round(2).to_string())

371 stocks with at least three years, 2012-2021:
                              Sharpe, average stock  worst drawdown, average stock %  annual return %  annual volatility %  average exposure
Hold                                           0.68                            47.61            18.95                29.18              1.00
Sized by trailing volatility                   0.71                            36.96            16.11                23.49              0.93
Sized by model volatility                      0.70                            36.03            15.51                22.82              0.90
Share of stocks whose Sharpe ratio improved on holding: {'Sized by trailing volatility': '64%', 'Sized by model volatility': '63%'}

Equal-weight portfolio of all stocks under each rule:
                              annual return %  annual volatility %  sharpe  worst drawdown %
Hold                                    18.67                17.31    1.08             38.73
Sized by trail

## 6. Findings

In [8]:
m0, m1, m2, m3 = (range_table.loc[k] for k in ('M0 trailing vol, normal', 'M1 trailing vol, past quantiles', 'M2 model vol, past quantiles', 'M3 model vol + VIX-tilted centre'))
c = confirm
print(f"1. Fear and return: {'CONFIRMED' if confirmed else 'NOT CONFIRMED'} on new periods. Rank correlation {c.loc['1990-2009 (new)', 'rank correlation']:+.2f} in 1990-2009 and "
      f"{c.loc['2022-2026 (new)', 'rank correlation']:+.2f} in 2022-2026 (pooled t {c.loc['both new periods pooled', 't']:.1f}), against {c.loc['2010-2021 (where it was found)', 'rank correlation']:+.2f} where it was found.")
print(f"   Over 1991-2026 the next-21-day return averaged {by_third.loc['fearful', 'mean next-21d return %']:.2f}% after fearful readings against {by_third.loc['calm', 'mean next-21d return %']:.2f}% after calm ones, "
      f"but the worst 5% of periods averaged {by_third.loc['fearful', 'worst 5% of periods, avg %']:.1f}% against {by_third.loc['calm', 'worst 5% of periods, avg %']:.1f}%.")
print(f"2. Volatility: the model's loss is {vol_table.loc['Model', 'QLIKE']:.3f} against {vol_table.loc['Trailing 20-day volatility', 'QLIKE']:.3f} for trailing volatility "
      f"(Diebold-Mariano {dm['dm']:.1f}); it ranks stocks by next-month volatility with IC {vol_table.loc['Model', 'rank IC within sector']:.2f} against {vol_table.loc['Trailing 20-day volatility', 'rank IC within sector']:.2f}.")
print(f"3. Honesty of the range: the original approach (M0) put {m0['90% band: inside %']:.1f}% of outcomes inside its 90% band and {m0['50% band: inside %']:.1f}% inside its 50% band. "
      f"The model with past quantiles (M2): {m2['90% band: inside %']:.1f}% and {m2['50% band: inside %']:.1f}%.")
print(f"   Interval score of the 90% band: M0 {m0['90% band: interval score']:.2f}, M1 {m1['90% band: interval score']:.2f}, M2 {m2['90% band: interval score']:.2f} (lower is better). "
      f"In fearful markets M0 covered {state_table.loc['M0 trailing vol, normal', 'fearful']:.1f}% and M2 {state_table.loc['M2 model vol, past quantiles', 'fearful']:.1f}%.")
print(f"4. Tilting the centre: out-of-sample R-squared {100 * oos_r2:+.2f}%; it changes the 90% interval score by {gain:+.3f}. The tilt is small beside the width of the range.")
s = strat_table
print(f"5. Sizing: average-stock Sharpe {s.loc['Hold', 'Sharpe, average stock']:.2f} holding, {s.loc['Sized by trailing volatility', 'Sharpe, average stock']:.2f} sized by trailing volatility, "
      f"{s.loc['Sized by model volatility', 'Sharpe, average stock']:.2f} sized by the model; worst drawdown {s.loc['Hold', 'worst drawdown, average stock %']:.0f}%, "
      f"{s.loc['Sized by trailing volatility', 'worst drawdown, average stock %']:.0f}%, {s.loc['Sized by model volatility', 'worst drawdown, average stock %']:.0f}%. "
      f"The model rule improved the Sharpe ratio of {better['Sized by model volatility']:.0f}% of stocks.")
print(f"   The rule holds {tension.loc['fearful', 'average exposure (model rule)']:.2f} of a position in fearful markets against {tension.loc['calm', 'average exposure (model rule)']:.2f} in calm ones, "
      f"while the market's next-21-day return was {tension.loc['fearful', 'market return over the next 21 days %']:.2f}% against {tension.loc['calm', 'market return over the next 21 days %']:.2f}%.")

J = lambda t: json.loads(t.astype(float).to_json(orient='index'))
summary = {'confirmed': confirmed, 'confirm': J(confirm), 'by_third': J(by_third), 'slope_2022': {'corr': float(slope['corr']), 't': float(slope['t'])},
           'vol': J(vol_table), 'vol_dm': dm, 'range': J(range_table), 'range_by_state': J(state_table),
           'centre': {'oos_r2': float(oos_r2), 'dm': dm_c, 'score_change': float(gain)},
           'strategy': J(strat_table), 'better_share': better, 'book': J(book_table), 'aapl': {k: json.loads(t.loc[TICKER].to_json()) for k, t in per_stock.items()},
           'aapl_inside_90': aapl_inside, 'tension': J(tension), 'oos': [FIRST_OOS, END], 'stocks': int(enough.sum())}
with open(os.path.join('data_cache', 'price_range_summary.json'), 'w') as f:
    json.dump(summary, f, indent=1)

1. Fear and return: CONFIRMED on new periods. Rank correlation +0.10 in 1990-2009 and +0.08 in 2022-2026 (pooled t 2.2), against +0.26 where it was found.
   Over 1991-2026 the next-21-day return averaged 1.10% after fearful readings against 0.57% after calm ones, but the worst 5% of periods averaged -14.6% against -6.0%.
2. Volatility: the model's loss is 0.427 against 0.578 for trailing volatility (Diebold-Mariano -8.1); it ranks stocks by next-month volatility with IC 0.51 against 0.46.
3. Honesty of the range: the original approach (M0) put 86.6% of outcomes inside its 90% band and 49.6% inside its 50% band. The model with past quantiles (M2): 88.2% and 48.6%.
   Interval score of the 90% band: M0 35.95, M1 36.22, M2 34.75 (lower is better). In fearful markets M0 covered 89.4% and M2 91.0%.
4. Tilting the centre: out-of-sample R-squared +0.87%; it changes the 90% interval score by +0.159. The tilt is small beside the width of the range.
5. Sizing: average-stock Sharpe 0.68 holding,

## 7. What this does not show

- **A range is not a direction.** Nothing here says whether a stock will rise or fall; it says how far it is likely to move.
- **The inputs were chosen on the scoring period.** The model's weights are always fitted on the past, but the five features were selected after seeing 2010–2021.
- **One decade for the stock-level tests**, 2012–2021, which was mostly a rising market; and companies that left the index are missing.
- **The fear-and-return test uses the price index**, without dividends, and overlapping 21-day windows; the t-statistic allows for the overlap but the number of truly separate sell-offs is small.
- **Sizing is tested without leverage and without financing costs or taxes.** A rule that only reduces exposure will usually lower both return and risk; the question is the ratio.
- **Ranges assume the next month resembles the past in distribution.** The quantiles come from history, so an event unlike anything in it will fall outside.